# iterative-probe — 2×2 の推移

変調範囲（`fc` / `stage4+fc`）× GroupDRO step size（1e-3 / 1e-2）の 4 run を読む。
seed 42、warmup 2 epoch → stage01 5 epoch → stage02 5 epoch、cohort 10 クラスタ。
run-id と条件の対応は [runs.md](runs.md)。

**図は変調範囲ごとに分ける。** アーキテクチャを図の単位に置くことで、線の色は step size だけを
表す。4 条件を 1 枚に重ねると線が交差して読めない。

run 記録の読み込みと集計は `collect.py` が持ち、この notebook は集計済みの表を読んで
分析と可視化だけを行う。

```bash
uv run python analysis/iterative-probe/collect.py <run-id> <run-id> <run-id> <run-id> \
  --baseline 20260921T103036Z-resnet-chexpert-s42-5538
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd

ROOT = Path.cwd() if Path("results").is_dir() else Path("analysis/iterative-probe")
RESULTS, FIGURES = ROOT / "results", ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

# 旧 repo から持ってきた図 style。Okabe–Ito の色循環と論文向けの軸設定を持つ。
matplotlib.style.use(ROOT.parent / "styles" / "fairness.mplstyle")
MUTED, GRID = "#52514e", "#dcdcd8"

frame = pd.read_csv(RESULTS / "epoch_metrics.csv")

# 色は step size に固定する。変調範囲は図そのものが表すので、色には載せない。
STEP_COLOR = {0.001: "#0173B2", 0.01: "#DE8F05"}
MODULATIONS = ["fc", "stage4+fc"]
STEPS = sorted(frame["step_size"].unique())

frame.groupby(["modulation", "step_size"])["run_id"].agg(["first", "count"])

## 指標の読み方

hidden cohort 系の指標は名前が似ているので、`hidden_cohort_logger.py:301` の定義で揃えておく。

| 列 | 定義 | 図の見出し |
|---|---|---|
| `val/hidden_min_auroc` | 10 cohort の AUROC の**最小**。いちばん悪い群の性能 | Worst-group AUROC |
| `val/hidden_auroc_gap` | 同じく**最大 − 最小**。群間がどれだけ開いているか | Cohort AUROC spread |
| `val/hidden_loss_gap` | cohort 平均 loss の最大 − 最小 | （図にしていない） |

spread は worst group の値ではなく**散らばりの幅**なので、小さいほど群間が揃っている。
worst が下がっても best がもっと下がれば spread は縮むため、2 つは別々に読む。

## 到達点

各 run の最終 epoch。`hidden_*` は cohort が存在する stage にだけあるので、warmup では空になる。

**注意**: cohort は stage ごとに、さらに run ごとに引き直される。hidden group は run 間でも
stage 間でも別物なので、群の同一性を前提にした読み方はできない。

In [ ]:
COLUMNS = {
    "val/auroc": "global AUROC",
    "val/bacc": "global bACC",
    "val/hidden_min_auroc": "worst AUROC",
    "val/hidden_min_bacc": "worst bACC",
    "val/hidden_auroc_gap": "cohort AUROC spread",
    "train/group_dro/weight_entropy": "weight entropy",
    "train/group_dro/max_q": "max q",
}

last = frame.sort_values("run_epoch").groupby(["modulation", "step_size"]).tail(1)
last = last.set_index([last["modulation"], last["step_size"]])[list(COLUMNS)].rename(columns=COLUMNS)
last.index.names = ["modulation", "step size"]
last.round(4)

## 描画の共通部分

横軸は run 全体の通し epoch。縦の区切りは stage の境目で、そこで cohort が引き直され、
GroupDRO の `q` も初期化される。seed は 42 の 1 本だけなので、旧 repo の図のような
seed 間のばらつき帯は引けない。

In [ ]:
def panel(axis, modulation, column, title, reference=None, reference_label=""):
    """1 つの panel に、指定した変調範囲の step size 別の推移を描く。

    Args:
        axis: 描画先
        modulation: 描く変調範囲
        column: 描く列
        title: panel の見出し
        reference: 水平の基準線。不要なら None
        reference_label: 基準線に添える名前

    Returns:
        None
    """
    data = frame[frame["modulation"] == modulation]
    if reference is not None:
        axis.axhline(reference, color=MUTED, linewidth=0.8, linestyle="--", zorder=1)
        axis.text(data["run_epoch"].max() + 0.3, reference, reference_label, color=MUTED, fontsize=8, va="center")
    for _, group in data.groupby("stage_index"):
        left = group["run_epoch"].min()
        if left > 0:
            axis.axvline(left - 0.5, color=GRID, linewidth=0.8, zorder=0)
    for step in STEPS:
        series = data[data["step_size"] == step]
        values = series[column]
        axis.plot(series["run_epoch"], values, color=STEP_COLOR[step], label=f"step {step:g}", zorder=3)
        valid = values.dropna()
        if not valid.empty:
            axis.annotate(f"{valid.iloc[-1]:.3f}", (series.loc[valid.index[-1], "run_epoch"], valid.iloc[-1]), textcoords="offset points", xytext=(6, 0), va="center", color=MUTED, fontsize=8)
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_xlim(-0.5, data["run_epoch"].max() + 2.0)


def figure_for(modulation, panels, columns=3):
    """変調範囲 1 つ分の figure を、指標ごとの panel を並べて作る。

    Args:
        modulation: 描く変調範囲
        panels: `(列名, 見出し)` または `(列名, 見出し, 基準線, 基準線の名前)` の並び
        columns: 1 行あたりの panel 数

    Returns:
        Figure: 描画した figure
    """
    rows = -(-len(panels) // columns)
    figure, axes = plt.subplots(rows, columns, figsize=(4.4 * columns, 3.6 * rows), squeeze=False)
    for axis, spec in zip(axes.ravel(), panels, strict=False):
        panel(axis, modulation, *spec)
    for axis in axes.ravel()[len(panels) :]:
        axis.set_visible(False)
    handles, labels = axes[0][0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=len(STEPS), bbox_to_anchor=(0.5, 1.05))
    figure.suptitle(f"modulation: {modulation}", y=1.10, fontsize=12)
    return figure


LEARNING_CURVES = [
    ("val/loss", "Validation loss"),
    ("val/auroc", "Validation AUROC"),
    ("val/bacc", "Validation balanced accuracy"),
    ("val/hidden_min_auroc", "Worst-group AUROC"),
    ("val/hidden_min_bacc", "Worst-group balanced accuracy"),
    ("val/hidden_auroc_gap", "Cohort AUROC spread (max - min)"),
]

## learning curves — fc

In [ ]:
figure = figure_for("fc", LEARNING_CURVES)
figure.savefig(FIGURES / "learning_curves_fc.png")

## learning curves — stage4+fc

In [ ]:
figure = figure_for("stage4+fc", LEARNING_CURVES)
figure.savefig(FIGURES / "learning_curves_stage4_fc.png")

## baseline — 通常の ResNet

反復学習の global 指標を、`hypernet_e2e` の通常 ResNet と突き合わせる。反復で何を失って
いるのかは、反復同士を比べても出てこない。

比較対象は `20260921T103036Z-resnet-chexpert-s42-5538`（ERM、30 epoch）。
[initial-resnet-vs-invariant](../initial-resnet-vs-invariant/) が invariant 化の対照に使っている
run と同じもので、`data_manifest.json` の train / val の sha256 が iterative 側と一致する。
optimizer（AdamW lr 1e-4 / wd 0.01）・batch size 128・class weight `[0.2014, 1.7986]`・seed 42・
transform も揃っているので、**global の val 指標はそのまま並べて読める**。

揃っていないのは学習の中身のほうになる。

- baseline は ResNet-50 の全体を 30 epoch、ERM で回す
- iterative は Spatial LoRA + metadata 条件付けを載せ、warmup 2 epoch のあと cohort ごとの
  GroupDRO へ切り替える。合計 12 epoch。backbone は凍結していないので、パラメータ数では
  baseline の上位集合になる

hidden cohort 系の指標は baseline に無い（cohort を引かないため）。比べられるのは global の
`val/auroc` / `val/bacc` / `val/loss` と、属性別の worst-group 指標までとする。

In [ ]:
baseline = pd.read_csv(RESULTS / "baseline_epoch_metrics.csv")

GLOBAL_COLUMNS = {
    "val/auroc": "global AUROC",
    "val/bacc": "global bACC",
    "val/loss": "val loss",
    "val/sex/worst_group_auroc": "sex worst AUROC",
    "val/race/worst_group_auroc": "race worst AUROC",
    "val/ethnicity/worst_group_auroc": "ethnicity worst AUROC",
    "val/age_group_65/worst_group_auroc": "age worst AUROC",
}
LAST_EPOCH = int(frame["run_epoch"].max())
BEST_EPOCH = int(baseline.loc[baseline["val/auroc"].idxmax(), "run_epoch"])


def snapshot(name, row):
    """1 epoch 分の行を、比較表の 1 行へ畳む。

    Args:
        name: 表に出す行の名前
        row: epoch 単位の表の 1 行

    Returns:
        dict: 行名と global 指標
    """
    return {"run": name, **{label: row[column] for column, label in GLOBAL_COLUMNS.items()}}


# baseline は 3 点で見る。iterative と同じ epoch 予算・checkpoint に選ばれる最良・30 epoch 後。
comparison = pd.DataFrame(
    [
        snapshot(f"ResNet ERM @ epoch {LAST_EPOCH}", baseline[baseline["run_epoch"] == LAST_EPOCH].iloc[0]),
        snapshot(f"ResNet ERM @ best AUROC (epoch {BEST_EPOCH})", baseline.loc[baseline["val/auroc"].idxmax()]),
        snapshot(f"ResNet ERM @ epoch {int(baseline['run_epoch'].max())} (final)", baseline.iloc[-1]),
        *[snapshot(f"iterative {condition} @ epoch {LAST_EPOCH}", data.sort_values("run_epoch").iloc[-1]) for condition, data in frame.groupby("condition")],
    ]
).set_index("run")
comparison.round(4)

### 推移を重ねる

baseline は 30 epoch 回しているので、iterative の 12 epoch を追い越した先まで描く。同じ
epoch 予算での位置（左半分）と、収束先（右端）を 1 枚で読むためになる。

In [ ]:
import matplotlib.patheffects as patheffects

GLOBAL_PANELS = [
    ("val/loss", "Validation loss"),
    ("val/auroc", "Validation AUROC"),
    ("val/bacc", "Validation balanced accuracy"),
]
BASELINE_COLOR = "#000000"


def against_baseline(modulation, panels=GLOBAL_PANELS):
    """iterative の推移に baseline の推移を重ねる。

    Args:
        modulation: 描く変調範囲
        panels: `(列名, 見出し)` の並び

    Returns:
        Figure: 描画した figure
    """
    figure, axes = plt.subplots(1, len(panels), figsize=(4.4 * len(panels), 3.6), squeeze=False)
    for axis, (column, title) in zip(axes[0], panels, strict=True):
        panel(axis, modulation, column, title)
        axis.plot(baseline["run_epoch"], baseline[column], color=BASELINE_COLOR, linewidth=1.2, linestyle=":", label="ResNet (ERM)", zorder=2)
        end = (baseline["run_epoch"].iloc[-1], baseline[column].iloc[-1])
        axis.annotate(f"{end[1]:.3f}", end, textcoords="offset points", xytext=(6, 0), va="center", color=BASELINE_COLOR, fontsize=8)
        axis.set_xlim(-0.5, baseline["run_epoch"].max() + 4.0)
        # 終端の数値は baseline の線と重なる位置に出ることがあるので、白で縁取って読めるようにする。
        for text in axis.texts:
            text.set_path_effects([patheffects.withStroke(linewidth=3.0, foreground="white")])
            text.set_zorder(5)
    handles, labels = axes[0][0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.09))
    figure.suptitle(f"modulation: {modulation}   (dotted: plain ResNet, ERM)", y=1.16, fontsize=12)
    return figure


for modulation in MODULATIONS:
    figure = against_baseline(modulation)
    figure.savefig(FIGURES / f"baseline_global_{modulation.replace('+', '_')}.png")

## 公平性 — demographic 群ごと

ここまでの図で見ていた `val/<属性>/worst_group_auroc` は、属性ごとの worst と gap しか残らない。
**群そのものの性能**も、**属性を掛け合わせた交差群**も run artifact には無いので、予測を cache して
群の切り方を分析側で決める。

```bash
uv run python analysis/iterative-probe/cache_predictions.py --split test
```

評価は **test split** で行う。各 run の checkpoint は val AUROC で選んでいるので、val で群別の
性能を比べると選択の効いた側に寄る。test は 5 model のどれも見ていない。この節の global 値が
上の節と一致しないのはそのためで、上は val、ここは test になる。

- **model**: baseline（ResNet ERM の best val AUROC）と iterative 4 本（`run.json` の
  `selected_checkpoint`＝最終 stage の best val AUROC）。選び方は揃えている
- **群**: age group（65 歳）× sex × race。race は **White / Asian / Black** に絞る。
  Other / Pacific Islander / Native American は test で n が小さく、交差させると評価が崩れる
- **母集団**: 3 属性すべてが非欠損で race が上の 3 つに入る行。どの粒度でも同じ 15,932 行を使うので、
  単独属性と交差群を並べて読める

`worst_group_auroc` の値がこれまでの図と違うのは、split（val → test）と race の絞り込みの両方が
効いているためになる。

In [ ]:
import json

import torch
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

CACHE, SPLIT = ROOT / "cache", "test"
REPOSITORY_ROOT = ROOT.parents[1]

# split CSV の符号。旧 repo の `create_cv_chexpert.py` が付けた対応で、race は 6 値のうち 3 つを使う。
AGE_LABELS = {0: "<65", 1: ">=65"}
SEX_LABELS = {0: "Male", 1: "Female"}
RACE_LABELS = {0: "White", 2: "Asian", 3: "Black"}
GROUPINGS = [("age",), ("sex",), ("race",), ("age", "sex"), ("age", "race"), ("sex", "race"), ("age", "sex", "race")]

split_frame = pd.read_csv(REPOSITORY_ROOT / "data" / "chexpert" / "splits" / f"{SPLIT}.csv")
# 年齢群は学習時と同じ定義（65 歳境界、`age_missing` はそのまま欠損）で作る。
age_missing = split_frame["age_missing"].astype(bool) | split_frame["age"].isna()
demographics = pd.DataFrame(
    {
        "age": (split_frame["age"] >= 65).map(lambda flag: AGE_LABELS[int(flag)]),
        "sex": split_frame["sex"].map(SEX_LABELS),
        "race": split_frame["race"].map(RACE_LABELS),
    }
)
valid = ~(age_missing | split_frame["sex_missing"].astype(bool) | split_frame["race_missing"].astype(bool)) & demographics.notna().all(axis=1)
demographics = demographics[valid]

BASELINE_RUN = baseline["run_id"].iloc[0]
MODELS = [{"run_id": BASELINE_RUN, "label": "ResNet (ERM)", "modulation": None, "step_size": None}] + [
    {"run_id": row.run_id, "label": f"{row.modulation} / {row.step_size:g}", "modulation": row.modulation, "step_size": row.step_size}
    for row in frame.drop_duplicates("run_id")[["run_id", "modulation", "step_size"]].itertuples()
]


def load_cache(run_id):
    """予測 cache を読み、split CSV と行が対応していることを確かめる。

    Args:
        run_id: cache を作った run の ID

    Returns:
        dict: `target` / `probabilities` / `predictions` と metadata
    """
    with np.load(CACHE / f"{run_id}_{SPLIT}.npz", allow_pickle=False) as cached:
        result = {key: cached[key] for key in cached.files}
    # cache は `evaluation_dataloader` の固定順＝CSV 順。ここが崩れると群の割り当てが全部ずれる。
    assert (result["image"] == split_frame["image"].to_numpy(dtype=str)).all()
    result["metadata"] = json.loads(str(result["metadata"]))
    return result


caches = {model["run_id"]: load_cache(model["run_id"]) for model in MODELS}
pd.DataFrame([{"model": m["label"], "checkpoint": caches[m["run_id"]]["metadata"]["checkpoint"]} for m in MODELS])

### 群ごとの性能

粒度ごとに 1 行 1 群の表を作る。`n` が小さい群では AUROC も TPR も揺れるので、群の値を読む前に
`n` と `positive_rate` を見る。3 属性の交差では最小の群が 118 行・陽性 9 件になる。

In [ ]:
def group_metrics(target, probability, prediction):
    """1 つの群の性能を返す。

    Args:
        target: 正解ラベル
        probability: 陽性クラスの確率
        prediction: 予測ラベル

    Returns:
        dict: `n` / `positive_rate` / `auroc` / `bacc` / `tpr` / `fpr`
    """
    positive, negative = target == 1, target == 0
    return {
        "n": len(target),
        "positive_rate": positive.mean(),
        "auroc": roc_auc_score(target, probability) if positive.any() and negative.any() else np.nan,
        "bacc": balanced_accuracy_score(target, prediction),
        "tpr": prediction[positive].mean() if positive.any() else np.nan,
        "fpr": prediction[negative].mean() if negative.any() else np.nan,
    }


def group_rows(model):
    """1 model 分を、全粒度 × 全群の行に展開する。

    Args:
        model: `MODELS` の 1 要素

    Returns:
        list[dict]: model・粒度・群名と性能
    """
    cached = caches[model["run_id"]]
    rows = []
    for keys in GROUPINGS:
        for values, part in demographics.groupby(list(keys), sort=True):
            # 行番号はそのまま cache の添字になる（cache は CSV 順、demographics は絞っただけ）。
            index = part.index.to_numpy()
            names = values if isinstance(values, tuple) else (values,)
            rows.append(
                {
                    "model": model["label"],
                    "grouping": " x ".join(keys),
                    "group": " / ".join(names),
                    **group_metrics(cached["target"][index], cached["probabilities"][index, 1], cached["predictions"][index]),
                }
            )
    return rows


groups_frame = pd.DataFrame([row for model in MODELS for row in group_rows(model)])
groups_frame.to_csv(RESULTS / f"group_metrics_{SPLIT}.csv", index=False)
groups_frame[groups_frame["model"] == "ResNet (ERM)"].set_index(["grouping", "group"]).drop(columns="model").round(4)

### 群間の差と、worst / best そのもの

gap だけでは worst が上がったのか best が下がったのかが読めないので、worst と best の値も並べる。
`Eopp1` は群間の TPR の最大差、`Eopp0` は TNR の最大差（FPR の最大差と同値）、`Eodds` は
`(TPR gap + FPR gap) / 2` で、`projects/*/utils/metrics.py` の定義と同じものになる。

In [ ]:
def summarize(part):
    """1 つの (model, 粒度) を、worst / best / gap の 1 行に畳む。

    Args:
        part: `groups_frame` の部分表

    Returns:
        pd.Series: worst・best・gap と Eopp0 / Eopp1 / Eodds
    """
    tpr_gap, fpr_gap = part["tpr"].max() - part["tpr"].min(), part["fpr"].max() - part["fpr"].min()
    return pd.Series(
        {
            "groups": len(part),
            "min n": int(part["n"].min()),
            "worst AUROC": part["auroc"].min(),
            "best AUROC": part["auroc"].max(),
            "AUROC gap": part["auroc"].max() - part["auroc"].min(),
            "worst bACC": part["bacc"].min(),
            "best bACC": part["bacc"].max(),
            "bACC gap": part["bacc"].max() - part["bacc"].min(),
            "Eopp1": tpr_gap,
            "Eopp0": part["fpr"].max() - part["fpr"].min(),
            "Eodds": (tpr_gap + fpr_gap) / 2,
        }
    )


GROUPING_ORDER = [" x ".join(keys) for keys in GROUPINGS]
summary = groups_frame.groupby(["grouping", "model"]).apply(summarize, include_groups=False)
summary = summary.reindex(pd.MultiIndex.from_product([GROUPING_ORDER, [model["label"] for model in MODELS]], names=["grouping", "model"]))
summary.to_csv(RESULTS / f"fairness_summary_{SPLIT}.csv")
summary.round(4)

`Eopp0` / `Eopp1` / `Eodds` は自前で群ごとの TPR・FPR から作っている。単独属性については
repo の `compute_fairness_metrics` と一致することを確かめておく（交差群はその関数では作れない）。

In [ ]:
import sys

if str(REPOSITORY_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT.resolve()))
from projects.hypernet_e2e.utils.metrics import compute_fairness_metrics  # noqa: E402

# 同じ母集団・同じ群で呼ぶために、絞り込んだ行だけを渡す。
codes = pd.DataFrame({"age": (split_frame["age"] >= 65).astype(int), "sex": split_frame["sex"], "race": split_frame["race"]})[valid]
cached = caches[BASELINE_RUN]
index = codes.index.to_numpy()
reference = compute_fairness_metrics(
    torch.from_numpy(cached["logits"][index]),
    torch.from_numpy(cached["target"][index]),
    {"categorical": torch.as_tensor(codes.to_numpy(), dtype=torch.long)},
    {"categorical": list(codes.columns)},
)
mine = summary.loc[(list(codes.columns), "ResNet (ERM)"), ["Eopp0", "Eopp1", "Eodds"]].droplevel("model")
pd.DataFrame(reference).T[["Eopp0", "Eopp1", "Eodds"]].sub(mine).abs().max().max()

### 図

色は上の節と同じ規約で、線の色が step size を、図が変調範囲を表す。baseline は step size を
持たないので黒（点線・四角）に固定し、条件の 1 つではなく参照線として置く。

1 枚目は粒度ごとの worst → best の幅。線が短いほど群間が揃っていて、線の位置が全体の水準になる。
2 枚目は 3 属性の交差 12 群を 1 群 1 行で並べる。

In [ ]:
BASELINE_STYLE = {"color": "#000000", "marker": "s", "linestyle": ":"}


def model_styles(modulation):
    """baseline と、指定した変調範囲の 2 条件を、描画順に返す。

    Args:
        modulation: 描く変調範囲

    Returns:
        list[tuple]: `(ラベル, 描画 kwargs)`
    """
    rows = [(model, BASELINE_STYLE if model["modulation"] is None else {"color": STEP_COLOR[model["step_size"]], "marker": "o", "linestyle": "-"}) for model in MODELS]
    return [(model["label"], style) for model, style in rows if model["modulation"] in (None, modulation)]


def fairness_ranges(modulation):
    """粒度ごとに worst → best の幅を描く。

    Args:
        modulation: 描く変調範囲

    Returns:
        Figure: 描画した figure
    """
    styles = model_styles(modulation)
    figure, axes = plt.subplots(1, 2, figsize=(7.0, 4.6), sharey=True)
    offsets = np.linspace(0.26, -0.26, len(styles))
    for axis, (metric, title) in zip(axes, [("auroc", "AUROC"), ("bacc", "balanced accuracy")], strict=True):
        for (label, style), offset in zip(styles, offsets, strict=True):
            for position, grouping in enumerate(GROUPING_ORDER):
                part = groups_frame[(groups_frame["model"] == label) & (groups_frame["grouping"] == grouping)][metric]
                axis.plot([part.min(), part.max()], [position + offset] * 2, color=style["color"], linewidth=1.6, solid_capstyle="round", zorder=3)
                axis.scatter([part.min(), part.max()], [position + offset] * 2, s=18, color=style["color"], marker=style["marker"], zorder=4)
            axis.plot([], [], label=label, **style)
        axis.set_yticks(range(len(GROUPING_ORDER)), GROUPING_ORDER)
        axis.set_title(f"worst - best {title}")
        axis.set_xlabel(title)
    # sharey なので反転は 1 回だけ。2 回呼ぶと元に戻る。
    axes[0].invert_yaxis()
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.06))
    figure.suptitle(f"modulation: {modulation}   ({SPLIT} split)", y=1.13, fontsize=12)
    return figure


def intersection_groups(modulation):
    """3 属性の交差 12 群を 1 群 1 行で描く。

    Args:
        modulation: 描く変調範囲

    Returns:
        Figure: 描画した figure
    """
    styles = model_styles(modulation)
    cells = groups_frame[groups_frame["grouping"] == "age x sex x race"]
    # 群の並びは baseline の AUROC 順に固定する。model ごとに並べ替えると行の対応が取れない。
    order = cells[cells["model"] == "ResNet (ERM)"].sort_values("auroc")["group"].tolist()
    labels = [f"{name}  (n={int(cells[(cells['model'] == 'ResNet (ERM)') & (cells['group'] == name)]['n'].iloc[0])})" for name in order]
    figure, axes = plt.subplots(1, 2, figsize=(7.6, 4.8), sharey=True)
    for axis, (metric, title) in zip(axes, [("auroc", "AUROC"), ("bacc", "balanced accuracy")], strict=True):
        for label, style in styles:
            part = cells[cells["model"] == label].set_index("group").loc[order, metric]
            axis.scatter(part.to_numpy(), range(len(order)), s=26, color=style["color"], marker=style["marker"], label=label, zorder=3)
        axis.set_yticks(range(len(order)), labels)
        axis.set_title(f"{title} per intersectional group")
        axis.set_xlabel(title)
    axes[0].invert_yaxis()
    handles, axis_labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, axis_labels, loc="upper center", ncol=len(axis_labels), bbox_to_anchor=(0.5, 1.06))
    figure.suptitle(f"modulation: {modulation}   ({SPLIT} split, age x sex x race)", y=1.12, fontsize=12)
    return figure


for modulation in MODULATIONS:
    suffix = modulation.replace("+", "_")
    fairness_ranges(modulation).savefig(FIGURES / f"fairness_ranges_{suffix}.png")
    intersection_groups(modulation).savefig(FIGURES / f"fairness_intersection_{suffix}.png")

## GroupDRO が動いたか

`weight_entropy` の上限は一様分布の log(10) = 2.3026。ここから離れるほど、特定の cohort へ
重みが寄っている。判定は「stage 内で寝るか（均衡）、下がり続けるか（未収束）」で行う。

In [ ]:
GROUP_DRO = [
    ("train/group_dro/weight_entropy", "Weight entropy", float(np.log(10)), "log(10)"),
    ("train/group_dro/max_q", "Max q", 0.1, "uniform"),
]

for modulation in MODULATIONS:
    figure = figure_for(modulation, GROUP_DRO, columns=2)
    figure.savefig(FIGURES / f"group_dro_{modulation.replace('+', '_')}.png")

## q は何に寄ったのか

cohort は stage ごとに引き直されるので、群を stage 間で追うことはできない。代わりに
**分析単位を `(run, stage, cohort)` とし、stage の中だけで `q` が何と相関するかを見る**。
知りたいのは「群 k がどうなったか」ではなく「DRO が何を難しさと見なしたか」なので、
stage 内で閉じた問いとして立てられる。4 run × 2 stage = 8 回の再抽選が、そのまま反復サンプルになる。

`assignments.parquet` は train / val / test を同じ `group_id` で持つので、train 側の `q_k` と
val 側の `hidden_auroc_k` は同じクラスタを指す。

対立仮説は 3 つ。

1. **重み由来** — `q` が陽性 class weight の大きい群に寄る。`weighting=inverse` では class weight が
   群の陽性率の決定的な関数なので、これは「陽性率の低い群に寄る」と同義
2. **難しさ由来** — `q` が AUROC の低い群に寄る。想定どおり
3. **サイズ由来** — `q` が小さい群に寄る。少数群は loss の分散が大きい

In [ ]:
groups = pd.read_csv(RESULTS / "cohort_groups.csv")

PAIRS = {
    "q ~ positive weight": "positive_weight",
    "q ~ val AUROC": "val_auroc",
    "q ~ train size": "train_size",
}
correlation = pd.DataFrame(
    [
        {
            "condition": condition,
            "stage": stage,
            **{name: data["q"].corr(data[column], method="spearman") for name, column in PAIRS.items()},
            "q spread": data["q"].max() - data["q"].min(),
        }
        for (condition, stage), data in groups.groupby(["condition", "stage"])
    ]
)
correlation.round(3)

## 全 cohort の推移

上の top3 / bottom3 は stage 最終 epoch の `q` で切った区分で、epoch ごとの順位変動を潰している。
圧縮せずに 10 群すべてを描く。

群の色は **stage 最終 epoch の `q` の順位**で決める（濃いほど `q` が大きい）。同じ図の中で同じ色は
同じ群を指すので、`q` の panel で上に行く線が AUROC の panel でどう動くかを追える。順位そのものは
色が表すので凡例は置かない。stage をまたぐと cohort が変わるため、色の対応も stage 内で閉じる。

群ごとの loss は今回の run には記録がない（`hidden_max_loss` と `hidden_loss_gap` のみ）。
`hidden_cohort_logger.py` に `hidden_loss_{k}` の記録を足したので、次の run からは同じ形で描ける。

In [ ]:
import matplotlib.cm as cm

# 群ごとに記録されうる指標。run に列が無いものは panel ごと落とす。bacc と loss は
# hidden_cohort_logger に後から足した記録なので、それ以前の run には存在しない。
GROUP_PANELS = [
    ("train/group_dro/q_{:02d}", "GroupDRO weight q", 0.1, "uniform"),
    ("val/hidden_auroc_{:02d}", "Validation AUROC per cohort", None, ""),
    ("val/hidden_bacc_{:02d}", "Validation bACC per cohort", None, ""),
    ("val/hidden_loss_{:02d}", "Validation loss per cohort", None, ""),
    ("val/hidden_support_{:02d}", "Validation support", None, ""),
]


def available_panels(data):
    """その run に記録がある panel だけを返す。

    Args:
        data: epoch 単位の表

    Returns:
        list: `GROUP_PANELS` の部分列
    """
    return [spec for spec in GROUP_PANELS if spec[0].format(0) in data.columns]


def all_groups(modulation, step):
    """10 群すべての推移を stage ごとに描く。色は stage 最終 epoch の q の順位。

    Args:
        modulation: 描く変調範囲
        step: 描く step size

    Returns:
        Figure: 描画した figure
    """
    stages = sorted(groups["stage"].unique())
    panels = available_panels(frame)
    figure, axes = plt.subplots(len(stages), len(panels), figsize=(4.6 * len(panels), 3.5 * len(stages)), squeeze=False)
    for row, stage in enumerate(stages):
        ranked = groups[(groups["modulation"] == modulation) & (groups["step_size"] == step) & (groups["stage"] == stage)].sort_values("q")
        # q の小さい群を薄く、大きい群を濃くする。順位は 1 つの sequential ramp で表す。
        shade = {group: cm.YlGnBu(0.25 + 0.7 * index / (len(ranked) - 1)) for index, group in enumerate(ranked["group"])}
        epochs = frame[(frame["modulation"] == modulation) & (frame["step_size"] == step) & (frame["stage"] == stage)]
        for column, (template, title, reference, reference_label) in enumerate(panels):
            axis = axes[row][column]
            if reference is not None:
                axis.axhline(reference, color=MUTED, linewidth=0.8, linestyle="--", zorder=1)
                axis.text(epochs["epoch"].max() + 0.05, reference, reference_label, color=MUTED, fontsize=8, va="center")
            for group in ranked["group"]:
                axis.plot(epochs["epoch"], epochs[template.format(group)], color=shade[group], linewidth=1.4, zorder=3)
            axis.set_title(f"{title} — {stage}")
            axis.set_xlabel("Stage epoch")
    figure.suptitle(f"modulation: {modulation} / step {step:g}   (color: darker = larger q)", y=1.02, fontsize=12)
    return figure


for modulation in MODULATIONS:
    for step in STEPS:
        figure = all_groups(modulation, step)
        figure.savefig(FIGURES / f"all_groups_{modulation.replace('+', '_')}_step{step:g}.png")